In [63]:
import streamlit as st
import pandas as pd
import numpy as np

from Updated_main_cost_engine import (
    load_interval_data,
    compute_interval_costs,
    compute_monthly_demand_charge,
    compute_baseline_annual_cost,
    TariffConfig,
    SPANISH_SAMPLE_TARIFF,
    scenario_reduce_overall,
    scenario_reduce_peak,
    scenario_shift_peak_energy,
)

# -----------------------------
# 1. PAGE SETUP
# -----------------------------
st.set_page_config(
    page_title="Real-time Energy Cost Management",
    layout="wide",
)

st.title("⚡ Real-time Energy Cost Management – Demo")
st.markdown(
    "Use your 15-minute interval load data + configurable tariff to explore "
    "monthly costs, daily behavior, and ROI of efficiency / peak-shaving actions."
)

# -----------------------------
# 2. LOAD DATA
# -----------------------------
with st.spinner("Loading interval data..."):
    load_df = load_interval_data()

st.success(
    f"Loaded {len(load_df):,} intervals "
    f"from {load_df.index.min().date()} to {load_df.index.max().date()}"
)

# -----------------------------
# 3. SIDEBAR – TARIFF + SCENARIOS
# -----------------------------
st.sidebar.header("Tariff configuration")

peak_price = st.sidebar.number_input(
    "Peak energy price [€/MWh]",
    value=float(SPANISH_SAMPLE_TARIFF.price_peak),
    step=5.0,
)
offpeak_price = st.sidebar.number_input(
    "Off-peak energy price [€/MWh]",
    value=float(SPANISH_SAMPLE_TARIFF.price_offpeak),
    step=5.0,
)
demand_price = st.sidebar.number_input(
    "Demand charge [€/kW·month]",
    value=float(SPANISH_SAMPLE_TARIFF.demand_price_eur_per_kw_month),
    step=1.0,
)
atr_price = st.sidebar.number_input(
    "ATR / regulated [€/MWh]",
    value=float(SPANISH_SAMPLE_TARIFF.atr_eur_per_mwh),
    step=1.0,
)
tax_rate = (
    st.sidebar.slider(
        "Taxes [% of subtotal]",
        min_value=0.0,
        max_value=25.0,
        value=float(SPANISH_SAMPLE_TARIFF.tax_rate * 100.0),
        step=0.5,
    )
    / 100.0
)

peak_start = st.sidebar.number_input(
    "Peak start hour", min_value=0, max_value=23,
    value=SPANISH_SAMPLE_TARIFF.peak_start_hour,
)
peak_end = st.sidebar.number_input(
    "Peak end hour", min_value=1, max_value=24,
    value=SPANISH_SAMPLE_TARIFF.peak_end_hour,
)

tariff = TariffConfig(
    name="UI_configured",
    price_peak=peak_price,
    price_offpeak=offpeak_price,
    demand_price_eur_per_kw_month=demand_price,
    atr_eur_per_mwh=atr_price,
    tax_rate=tax_rate,
    peak_start_hour=peak_start,
    peak_end_hour=peak_end,
)

st.sidebar.markdown("---")
st.sidebar.header("Scenario parameters")

overall_reduction = (
    st.sidebar.slider(
        "Overall consumption reduction [%]",
        0.0, 30.0, 5.0, 1.0,
    )
    / 100.0
)

peak_reduction_kw = st.sidebar.number_input(
    "Peak-shaving capability [kW]", value=200.0, step=50.0
)

shift_fraction = (
    st.sidebar.slider(
        "Shift peak energy to off-peak [%]",
        0.0, 50.0, 15.0, 5.0,
    )
    / 100.0
)

capex_eff = st.sidebar.number_input(
    "CAPEX – efficiency [€]", value=30_000.0, step=5_000.0
)
capex_peak = st.sidebar.number_input(
    "CAPEX – peak-shaving [€]", value=80_000.0, step=5_000.0
)
capex_shift = st.sidebar.number_input(
    "CAPEX – shifting [€]", value=10_000.0, step=2_000.0
)

# -----------------------------
# 4. COST ENGINE (BASELINE)
# -----------------------------
with st.spinner("Computing interval costs with current tariff..."):
    cost_df = compute_interval_costs(load_df, tariff)
    cost_df = compute_monthly_demand_charge(cost_df, tariff)

st.subheader("🔎 Data preview (first 10 intervals)")
st.dataframe(cost_df.head(10))

# -----------------------------
# 5. MONTHLY COST BREAKDOWN
# -----------------------------
st.subheader("📊 Monthly cost breakdown")

monthly_energy = cost_df["energy_cost_eur_interval"].resample("ME").sum()
monthly_atr = cost_df["atr_cost_eur_interval"].resample("ME").sum()
monthly_demand = cost_df.groupby(pd.Grouper(freq="ME"))[
    "demand_charge_eur_month"
].max()

monthly_cost = pd.concat(
    [monthly_energy, monthly_atr, monthly_demand],
    axis=1,
)
monthly_cost.columns = ["energy_cost_eur", "atr_cost_eur", "demand_charge_eur"]

st.bar_chart(monthly_cost)

st.markdown(
    f"**Total annual cost (last 12 months):** "
    f"{compute_baseline_annual_cost(cost_df):,.0f} €"
)

# -----------------------------
# 6. DAILY LOAD & COST VIEW
# -----------------------------
st.subheader("📅 Daily load & cost")

min_date = cost_df.index.min().date()
max_date = cost_df.index.max().date()

selected_date = st.date_input(
    "Select a day",
    value=min_date,
    min_value=min_date,
    max_value=max_date,
)

day_start = pd.Timestamp(selected_date)
day_end = day_start + pd.Timedelta(days=1)
day_data = cost_df.loc[day_start:day_end]

if day_data.empty:
    st.info("No data for this day.")
else:
    st.markdown("**Load profile [kW]**")
    st.line_chart(day_data[["load_kw"]], height=250)

    st.markdown("**Interval cost [€]**")
    st.line_chart(day_data[["total_cost_eur_interval"]], height=250)

# -----------------------------
# 7. ACTION SCENARIOS (ROI)
# -----------------------------
st.subheader("💡 Optimization scenarios – savings & payback")

scenarios = []
try:
    scenarios.append(
        scenario_reduce_overall(
            cost_df,
            tariff,
            reduction_pct=overall_reduction,
            capex_eur=capex_eff,
        )
    )
    scenarios.append(
        scenario_reduce_peak(
            cost_df,
            tariff,
            peak_reduction_kw=peak_reduction_kw,
            capex_eur=capex_peak,
        )
    )
    scenarios.append(
        scenario_shift_peak_energy(
            cost_df,
            tariff,
            shift_fraction=shift_fraction,
            capex_eur=capex_shift,
        )
    )
except Exception as e:
    st.error(f"Error while computing scenarios: {e}")
    scenarios = []

if scenarios:
    actions_df = pd.DataFrame([s.__dict__ for s in scenarios])
    actions_df = actions_df.sort_values("roi_simple_payback_years")

    st.dataframe(
        actions_df.style.format(
            {
                "capex_eur": "{:,.0f}",
                "savings_eur_per_year": "{:,.0f}",
                "roi_simple_payback_years": "{:,.1f}",
            }
        )
    )

    best = actions_df.iloc[0]
    st.markdown(
        f"👉 **Best scenario right now:** `{best['name']}`  \n"
        f"• CAPEX: **{best['capex_eur']:,.0f} €**  \n"
        f"• Annual savings: **{best['savings_eur_per_year']:,.0f} €**  \n"
        f"• Simple payback: **{best['roi_simple_payback_years']:.1f} years**"
    )
else:
    st.info("No scenarios computed (see error above).")